In [1]:
# ==========================================
# 1. SETUP & IMPORTS
# ==========================================
from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer, OneHotEncoder
from pyspark.ml.regression import GBTRegressor, RandomForestRegressor, GeneralizedLinearRegression
import os

# Initialize Spark
spark = (SparkSession.builder
    .appName("BIXI-Model-Factory")
    .config("spark.driver.memory", "4g")  # Increase Driver memory to 4GB
    .config("spark.executor.memory", "4g") # Increase Executor memory to 4GB
    .config("spark.sql.session.timeZone", "America/Toronto")
    .getOrCreate())


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/18 01:11:44 WARN Utils: Your hostname, Comanes-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.11 instead (on interface en0)
26/04/18 01:11:44 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/18 01:11:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
# ==========================================
# 2. CONFIGURATION (Change this for new stations)
# ==========================================
targetStation = "STN_0001"
gold_path = "data/gold/station_flow"
models_save_dir = f"models/{targetStation}/"

# Ensure save directory exists
os.makedirs(models_save_dir, exist_ok=True)



In [3]:
# ==========================================
# 3. DATA LOADING & TEMPORAL SPLIT
# ==========================================
print(f"Loading data for {targetStation}...")
gold_df = spark.read.parquet(gold_path).filter(F.col("station_id") == targetStation)

# Hardcoded temporal split based on business requirements
split_date = "2025-08-01 00:00:00"

# For the factory, we ONLY need the training data (< 2025-08-01)
train_df = gold_df.filter(F.col("ts_hour") < split_date).cache()
print(f"Training Data Ready: {train_df.count():,} rows. Cutoff date: strictly before {split_date}")


Loading data for STN_0001...


26/04/18 01:11:47 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Training Data Ready: 20,197 rows. Cutoff date: strictly before 2025-08-01 00:00:00


In [4]:
# ==========================================
# 4. FEATURE DEFINITIONS
# ==========================================
EXCLUDE_COLS = ["ts_hour", "station_id", "station_inflow", "station_outflow", "station_netflow"]

# Define numeric and categorical columns
categorical_cols = ["temp_bin"] # Add any other string columns here
numeric_cols = [f.name for f in train_df.schema.fields 
                if f.dataType.typeName() in ['integer', 'long', 'double', 'float'] 
                and f.name not in EXCLUDE_COLS]

# Create standard feature stages
feature_stages = []
for col in categorical_cols:
    feature_stages.append(StringIndexer(inputCol=col, outputCol=f"{col}_idx", handleInvalid="keep"))
    feature_stages.append(OneHotEncoder(inputCol=f"{col}_idx", outputCol=f"{col}_vec"))

encoded_cols = [f"{col}_vec" for col in categorical_cols]
assembler = VectorAssembler(inputCols=numeric_cols + encoded_cols, outputCol="features", handleInvalid="skip")
feature_stages.append(assembler)

# Add a scaler specifically for the Poisson GLR
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures", withStd=True, withMean=False)



In [5]:
# ==========================================
# 5. MODEL ARCHITECTURES & TUNING GRIDS
# ==========================================
from pyspark.ml.regression import GBTRegressor, RandomForestRegressor, GeneralizedLinearRegression
from pyspark.ml.tuning import ParamGridBuilder

# Define the base models 
gbt = GBTRegressor(featuresCol="features", seed=42)
rf = RandomForestRegressor(featuresCol="features", seed=42)
poisson = GeneralizedLinearRegression(featuresCol="scaledFeatures", family="poisson", link="log")

# Build the grids
gbt_grid = (ParamGridBuilder()
    .addGrid(gbt.maxDepth, [4, 6, 8])          
    .addGrid(gbt.maxIter, [50, 100])        
    .build())

rf_grid = (ParamGridBuilder()
    .addGrid(rf.maxDepth, [5, 8, 10])      
    .addGrid(rf.numTrees, [50, 100])        
    .build())

poisson_grid = (ParamGridBuilder()
    .addGrid(poisson.regParam, [0.01, 0.1]) 
    .build())

# THE CRITICAL FIX: The nested dictionary
architectures = {
    "GBT": {"model": gbt, "grid": gbt_grid},
    "RF": {"model": rf, "grid": rf_grid},
    "Poisson": {"model": poisson, "grid": poisson_grid}
}

In [6]:
# ==========================================
# 6. THE BASIC TRAINING LOOP
# ==========================================

# 1. Define the missing targets dictionary
targets = {
    "outflow": "station_outflow",
    "inflow": "station_inflow"
}

for flow_type, target_col in targets.items():
    print(f"\n--- Training {flow_type.upper()} Models ---")
    
    for model_name, config in architectures.items():
        print(f"Training {model_name}...")
        
        # 2. Access the model object from the nested dictionary
        algo = config["model"]
        
        # Set the target column dynamically
        algo.setLabelCol(target_col)
        
        # Build Pipeline (Poisson needs the scaler)
        if model_name == "Poisson":
            pipeline = Pipeline(stages=feature_stages + [scaler, algo])
        else:
            pipeline = Pipeline(stages=feature_stages + [algo])
            
        # Fit the model
        trained_model = pipeline.fit(train_df)
        
        # Save the model to disk (overwrite if exists)
        save_path = os.path.join(models_save_dir, f"{flow_type}_{model_name}")
        trained_model.write().overwrite().save(save_path)
        print(f"✅ Saved to: {save_path}")

print("\n🎉 All models for STN_0001 trained and saved successfully!")


--- Training OUTFLOW Models ---
Training GBT...
✅ Saved to: models/STN_0001/outflow_GBT
Training RF...
✅ Saved to: models/STN_0001/outflow_RF
Training Poisson...


26/04/18 01:11:55 WARN Instrumentation: [d98eef3c] regParam is zero, which might cause numerical instability and overfitting.
26/04/18 01:11:55 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/04/18 01:11:55 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK
26/04/18 01:11:55 WARN Instrumentation: [d98eef3c] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/18 01:11:56 WARN Instrumentation: [d98eef3c] regParam is zero, which might cause numerical instability and overfitting.
26/04/18 01:11:56 WARN Instrumentation: [d98eef3c] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/18 01:11:56 WARN Instrumentation: [d98eef3c] regParam is zero, which might cause numerical instability and overfitting.
26/04/18 01:11:56 WARN Instrumentation: [d98eef3c] Cholesky solver failed due to singular covariance matrix. Retryin

✅ Saved to: models/STN_0001/outflow_Poisson

--- Training INFLOW Models ---
Training GBT...
✅ Saved to: models/STN_0001/inflow_GBT
Training RF...
✅ Saved to: models/STN_0001/inflow_RF
Training Poisson...


26/04/18 01:12:03 WARN Instrumentation: [ca3d9272] regParam is zero, which might cause numerical instability and overfitting.
26/04/18 01:12:03 WARN Instrumentation: [ca3d9272] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/18 01:12:03 WARN Instrumentation: [ca3d9272] regParam is zero, which might cause numerical instability and overfitting.
26/04/18 01:12:03 WARN Instrumentation: [ca3d9272] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/18 01:12:03 WARN Instrumentation: [ca3d9272] regParam is zero, which might cause numerical instability and overfitting.
26/04/18 01:12:04 WARN Instrumentation: [ca3d9272] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/18 01:12:04 WARN Instrumentation: [ca3d9272] regParam is zero, which might cause numerical instability and overfitting.
26/04/18 01:12:04 WARN Instrumentation: [ca3d9272] Cholesky solv

✅ Saved to: models/STN_0001/inflow_Poisson

🎉 All models for STN_0001 trained and saved successfully!


In [7]:
# ==========================================
# 7. TIME-SERIES HYPERPARAMETER TUNING 
# ==========================================

from pyspark.ml.evaluation import RegressionEvaluator

# We split the training data AGAIN to create a strict "Validation" period
# Example: Let's use the month of July 2025 as the tuning validation set
validation_date = "2025-07-01 00:00:00"

# Train the tuning models on everything before July
tune_train_df = train_df.filter(F.col("ts_hour") < validation_date).cache()
# Evaluate the tuning models strictly on July
tune_val_df = train_df.filter(F.col("ts_hour") >= validation_date).cache()

print(f"Tuning Train Size: {tune_train_df.count():,} | Tuning Val Size (July): {tune_val_df.count():,}")

evaluator = RegressionEvaluator(metricName="rmse")

for flow_type, target_col in targets.items():
    print(f"\n--- Tuning {flow_type.upper()} Models ---")
    evaluator.setLabelCol(target_col)
    
    for model_name, components in architectures.items():
        print(f"Starting Time-Series Grid Search for {model_name}...")
        
        algo = components["model"]
        grid = components["grid"]
        algo.setLabelCol(target_col)
        
        best_rmse = float("inf")
        best_params = None
        
        # Loop through every combination of parameters Spark generated
        for param_map in grid:
            # Apply the parameters to the algorithm
            algo_with_params = algo.copy(param_map)
            
            # Build temporary pipeline
            if model_name == "Poisson":
                pipeline = Pipeline(stages=feature_stages + [scaler, algo_with_params])
            else:
                pipeline = Pipeline(stages=feature_stages + [algo_with_params])
            
            # Train on the past (< July)
            temp_model = pipeline.fit(tune_train_df)
            
            # Predict on the future (July)
            predictions = temp_model.transform(tune_val_df)
            current_rmse = evaluator.evaluate(predictions)
            
            # Keep track of the lowest error
            if current_rmse < best_rmse:
                best_rmse = current_rmse
                best_params = param_map
                
        print(f"🏆 Best {model_name} Params found! RMSE: {best_rmse:.3f}")
        
        # FINAL STEP: Retrain the absolute best model on the FULL training set (< August)
        print(f"Retraining final {model_name} on all training data...")
        final_algo = algo.copy(best_params)
        final_pipeline = Pipeline(stages=feature_stages + ([scaler, final_algo] if model_name == "Poisson" else [final_algo]))
        
        best_final_model = final_pipeline.fit(train_df)
        
        # Save to disk
        save_path = os.path.join(models_save_dir, f"{flow_type}_{model_name}_optimized")
        best_final_model.write().overwrite().save(save_path)
        print(f"✅ Saved to: {save_path}")

Tuning Train Size: 19,453 | Tuning Val Size (July): 744

--- Tuning OUTFLOW Models ---
Starting Time-Series Grid Search for GBT...


26/04/18 01:12:37 WARN DAGScheduler: Broadcasting large task binary with size 1002.6 KiB
26/04/18 01:12:37 WARN DAGScheduler: Broadcasting large task binary with size 1005.8 KiB
26/04/18 01:12:37 WARN DAGScheduler: Broadcasting large task binary with size 1006.3 KiB
26/04/18 01:12:37 WARN DAGScheduler: Broadcasting large task binary with size 1007.3 KiB
26/04/18 01:12:37 WARN DAGScheduler: Broadcasting large task binary with size 1008.0 KiB
26/04/18 01:12:37 WARN DAGScheduler: Broadcasting large task binary with size 1010.1 KiB
26/04/18 01:12:37 WARN DAGScheduler: Broadcasting large task binary with size 1013.9 KiB
26/04/18 01:12:37 WARN DAGScheduler: Broadcasting large task binary with size 1016.7 KiB
26/04/18 01:12:37 WARN DAGScheduler: Broadcasting large task binary with size 1017.2 KiB
26/04/18 01:12:37 WARN DAGScheduler: Broadcasting large task binary with size 1018.2 KiB
26/04/18 01:12:37 WARN DAGScheduler: Broadcasting large task binary with size 1018.9 KiB
26/04/18 01:12:37 WAR

🏆 Best GBT Params found! RMSE: 7.178
Retraining final GBT on all training data...
✅ Saved to: models/STN_0001/outflow_GBT_optimized
Starting Time-Series Grid Search for RF...


26/04/18 01:13:41 WARN DAGScheduler: Broadcasting large task binary with size 1463.2 KiB
26/04/18 01:13:41 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/04/18 01:13:44 WARN DAGScheduler: Broadcasting large task binary with size 1458.3 KiB
26/04/18 01:13:45 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/04/18 01:13:46 WARN DAGScheduler: Broadcasting large task binary with size 4.9 MiB
26/04/18 01:13:48 WARN DAGScheduler: Broadcasting large task binary with size 1162.4 KiB
26/04/18 01:13:50 WARN DAGScheduler: Broadcasting large task binary with size 1463.2 KiB
26/04/18 01:13:51 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/04/18 01:13:51 WARN DAGScheduler: Broadcasting large task binary with size 4.7 MiB
26/04/18 01:13:52 WARN DAGScheduler: Broadcasting large task binary with size 1056.2 KiB
26/04/18 01:13:53 WARN DAGScheduler: Broadcasting large task binary with size 8.3 MiB
26/04/18 01:13:54 WARN DAGScheduler: Br

🏆 Best RF Params found! RMSE: 7.236
Retraining final RF on all training data...


...

🏆 Best GBT Params found! RMSE: 6.311
Retraining final GBT on all training data...
✅ Saved to: models/STN_0001/inflow_GBT_optimized
Starting Time-Series Grid Search for RF...


26/04/18 01:16:00 WARN DAGScheduler: Broadcasting large task binary with size 1463.3 KiB
26/04/18 01:16:01 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/04/18 01:16:04 WARN DAGScheduler: Broadcasting large task binary with size 1458.1 KiB
26/04/18 01:16:05 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/04/18 01:16:07 WARN DAGScheduler: Broadcasting large task binary with size 4.9 MiB
26/04/18 01:16:08 WARN DAGScheduler: Broadcasting large task binary with size 1166.4 KiB
26/04/18 01:16:10 WARN DAGScheduler: Broadcasting large task binary with size 1463.3 KiB
26/04/18 01:16:11 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/04/18 01:16:12 WARN DAGScheduler: Broadcasting large task binary with size 4.7 MiB
26/04/18 01:16:13 WARN DAGScheduler: Broadcasting large task binary with size 1050.1 KiB
26/04/18 01:16:13 WARN DAGScheduler: Broadcasting large task binary with size 8.2 MiB
26/04/18 01:16:14 WARN DAGScheduler: Br

🏆 Best RF Params found! RMSE: 6.341
Retraining final RF on all training data...


26/04/18 01:16:31 WARN DAGScheduler: Broadcasting large task binary with size 1430.7 KiB
26/04/18 01:16:32 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/04/18 01:16:34 WARN DAGScheduler: Broadcasting large task binary with size 4.9 MiB
26/04/18 01:16:35 WARN DAGScheduler: Broadcasting large task binary with size 1176.6 KiB
26/04/18 01:16:35 WARN DAGScheduler: Broadcasting large task binary with size 7.7 MiB
26/04/18 01:16:36 WARN DAGScheduler: Broadcasting large task binary with size 1770.7 KiB
26/04/18 01:16:37 WARN DAGScheduler: Broadcasting large task binary with size 8.5 MiB
26/04/18 01:16:38 WARN DAGScheduler: Broadcasting large task binary with size 1769.1 KiB
26/04/18 01:16:39 WARN DAGScheduler: Broadcasting large task binary with size 7.1 MiB
26/04/18 01:16:40 WARN DAGScheduler: Broadcasting large task binary with size 1485.9 KiB
26/04/18 01:16:41 WARN DAGScheduler: Broadcasting large task binary with size 3.0 MiB
26/04/18 01:16:42 WARN TaskSetManager: 

✅ Saved to: models/STN_0001/inflow_RF_optimized
Starting Time-Series Grid Search for Poisson...
🏆 Best Poisson Params found! RMSE: 9.853
Retraining final Poisson on all training data...
✅ Saved to: models/STN_0001/inflow_Poisson_optimized


In [8]:
#########
# COMPARISON TABLE OF ALL MODELS
#########

import pandas as pd
import os
from pyspark.ml import PipelineModel
from pyspark.ml.evaluation import RegressionEvaluator

# 1. Setup our three mathematical evaluators
rmse_evaluator = RegressionEvaluator(metricName="rmse")
mae_evaluator = RegressionEvaluator(metricName="mae")
r2_evaluator = RegressionEvaluator(metricName="r2")

# Create an empty list to hold our results
evaluation_results = []

print("Evaluating saved models... This might take a moment.")

# 2. Loop through both Inflow and Outflow
for flow_type, target_col in targets.items():
    
    # Tell the evaluators which column represents the "Truth"
    rmse_evaluator.setLabelCol(target_col)
    mae_evaluator.setLabelCol(target_col)
    r2_evaluator.setLabelCol(target_col)
    
    # 3. Loop through the three architectures we trained
    for model_name in ["GBT", "RF", "Poisson"]:
        
        # Construct the path to where the model is saved
        model_path = os.path.join(models_save_dir, f"{flow_type}_{model_name}_optimized")
        
        try:
            # Load the optimized pipeline from disk
            loaded_model = PipelineModel.load(model_path)
            
            # Make predictions on your validation/test set 
            # Note: Replace 'tune_val_df' with whatever dataframe you want to test on!
            predictions = loaded_model.transform(tune_val_df)
            
            # Calculate the metrics
            rmse = rmse_evaluator.evaluate(predictions)
            mae = mae_evaluator.evaluate(predictions)
            r2 = r2_evaluator.evaluate(predictions)
            
            # Store the results
            evaluation_results.append({
                "Flow Type": flow_type.upper(),
                "Model": model_name,
                "RMSE": round(rmse, 3),
                "MAE": round(mae, 3),
                "R-Squared": round(r2, 3)
            })
            
        except Exception as e:
            print(f"Could not evaluate {flow_type} {model_name}. Did it save correctly? Error: {e}")

# 4. Convert the results into a clean Pandas DataFrame for visualization
comparison_df = pd.DataFrame(evaluation_results)

# Sort the table so it's easy to read: Group by Flow Type, then rank by best MAE
comparison_df = comparison_df.sort_values(by=["Flow Type", "MAE"], ascending=[True, True]).reset_index(drop=True)

print("\n🏆 Model Comparison Table 🏆")
display(comparison_df) # Use print(comparison_df) if 'display' doesn't work in your environment

Evaluating saved models... This might take a moment.

🏆 Model Comparison Table 🏆


,Flow Type,Model,RMSE,MAE,R-Squared
0,INFLOW,RF,4.340,3.253,0.931
1,INFLOW,GBT,5.063,3.737,0.905
2,INFLOW,Poisson,9.444,6.591,0.671
3,OUTFLOW,RF,4.937,3.733,0.927
4,OUTFLOW,GBT,5.584,4.229,0.906
5,OUTFLOW,Poisson,10.623,7.278,0.660


# outflow model (RF) optimisation

In [9]:
#########
# FEATURE IMPORTANCE ANALYSIS (For the best RF model)
#########

import pandas as pd
from pyspark.ml import PipelineModel

import pandas as pd
from pyspark.ml import PipelineModel

# 1. Load the model
model_path = f"models/{targetStation}/outflow_RF_optimized"
print(f"Loading model from: {model_path}...")
loaded_pipeline = PipelineModel.load(model_path)
ml_model = loaded_pipeline.stages[-1]

# 2. Extract the importances array
importances = ml_model.featureImportances.toArray()

# 3. THE FIX: Extract true feature names from PySpark metadata
# We push 1 row of data through the pipeline to generate the metadata
# Note: Using 'tune_val_df' here, but you can use 'train_df' or any df in memory
sample_transformed = loaded_pipeline.transform(tune_val_df.limit(1))

# Get the exact column the model used (usually "features")
features_col_name = ml_model.getFeaturesCol()

# Dig into PySpark's hidden metadata dictionary
feature_attrs = sample_transformed.schema[features_col_name].metadata["ml_attr"]["attrs"]

# Unpack the metadata into a list of (index, name) tuples
extracted_names = []
for attr_group in feature_attrs.values():
    for attr in attr_group:
        extracted_names.append((attr['idx'], attr['name']))

# Sort strictly by index so it matches the importances array perfectly!
extracted_names.sort(key=lambda x: x[0])
true_feature_names = [name for idx, name in extracted_names]

# 4. Build a clean Pandas DataFrame (Now they will be the exact same length!)
importance_df = pd.DataFrame({
    "Feature": true_feature_names,
    "Importance": importances
})

# 5. Calculate the percentage and display
importance_df["Importance_%"] = (importance_df["Importance"] * 100).round(2)
importance_df = importance_df.sort_values(by="Importance", ascending=False).reset_index(drop=True)

print("\n🏆 Top 10 Most Important Features:")
display(importance_df.head(10))

print("\n🗑️ The 'Noise' (Bottom 15 Features):")
display(importance_df.tail(15))

Loading model from: models/STN_0001/outflow_RF_optimized...

🏆 Top 10 Most Important Features:


,Feature,Importance,Importance_%
0,radius500m_outflow_lag1,0.267239,26.72
1,radius100m_inflow_lag1,0.150469,15.05
2,radius200m_outflow_lag1,0.143327,14.33
3,station_inflow_lag1,0.065328,6.53
4,hod,0.040210,4.02
5,hod_cos,0.038712,3.87
6,radius100m_outflow_lag1,0.034636,3.46
7,radius200m_inflow_lag1,0.032635,3.26
8,radius500m_inflow_lag1,0.024723,2.47
9,station_outflow_rollsum6,0.020919,2.09



🗑️ The 'Noise' (Bottom 15 Features):


,Feature,Importance,Importance_%
52,station_inflow_rollsum6,0.001228,0.12
53,moy,0.001061,0.11
54,radius200m_inflow_rollsum6,0.000551,0.06
55,precip_rollmean3,0.000418,0.04
56,precip,0.000417,0.04
57,precip_rollsum3,0.000387,0.04
58,temp_bin_vec_15:20,0.000204,0.02
59,temp_bin_vec_0:10,0.000141,0.01
60,temp_bin_vec_20:25,0.000140,0.01
61,temp_bin_vec_10:15,0.000120,0.01


In [10]:
# Extract the top 15 feature names from your ranked dataframe
top_15_features = importance_df["Feature"].head(15).tolist()

print("🎯 Your Top 15 Winning Features:")
print("TOP_15_COLS =", top_15_features)

🎯 Your Top 15 Winning Features:
TOP_15_COLS = ['radius500m_outflow_lag1', 'radius100m_inflow_lag1', 'radius200m_outflow_lag1', 'station_inflow_lag1', 'hod', 'hod_cos', 'radius100m_outflow_lag1', 'radius200m_inflow_lag1', 'radius500m_inflow_lag1', 'station_outflow_rollsum6', 'station_outflow_lag1', 'radius100m_outflow_rollsum6', 'radius100m_outflow_rollmean6', 'radius500m_inflow_lag12', 'station_outflow_rollmean6']


In [11]:
# ==========================================
# STRICT FEATURE DEFINITIONS (Pruned to Top 15)
# ==========================================

# Paste the list you generated in Step 1 right here:
TOP_15_COLS = [
    "radius500m_outflow_lag1", 
    "radius100m_inflow_lag1", 
    "radius200m_outflow_lag1", 
    "station_inflow_lag1", 
    "hod", 
    "hod_cos", 
    "radius100m_outflow_lag1", 
    "radius200m_inflow_lag1", 
    "radius500m_inflow_lag1", 
    "station_outflow_rollsum6", 
    "station_outflow_lag1", 
    "radius100m_outflow_rollsum6", 
    "radius100m_outflow_rollmean6", 
    "radius500m_inflow_lag12", 
    "station_outflow_rollmean6"
    
]

# Create standard feature stages
feature_stages = []

# NOTE: If any of your categorical columns made the Top 15, 
# you still need to StringIndex and OneHotEncode them here!
categorical_cols = ["temp_bin"] # Keep this if temp_bin is in your Top 15

for col in categorical_cols:
    feature_stages.append(StringIndexer(inputCol=col, outputCol=f"{col}_idx", handleInvalid="keep"))
    feature_stages.append(OneHotEncoder(inputCol=f"{col}_idx", outputCol=f"{col}_vec"))

# Build the final VectorAssembler strictly from your top 15 list
assembler = VectorAssembler(inputCols=TOP_15_COLS, outputCol="features", handleInvalid="skip")
feature_stages.append(assembler)

# Add the scaler for the Poisson GLR
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures", withStd=True, withMean=False)

In [12]:
# ==========================================
# 1. SETUP & STRICT TEMPORAL SPLITS
# ==========================================
import os
import pyspark.sql.functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.tuning import ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator

targetStation = "STN_0001" # Change if testing a different station
target_col = "station_outflow"

# Ensure gold_df is loaded beforehand, then filter to the target station
gold_df_stn = gold_df.filter(F.col("station_id") == targetStation)

# Time bounds (Train < Aug 1st, Test >= Aug 1st)
test_date = "2025-08-01 00:00:00"
validation_date = "2025-07-01 00:00:00" # Carving out July for tuning

# Create splits
train_full_df = gold_df_stn.filter(F.col("ts_hour") < test_date).cache()
test_df = gold_df_stn.filter(F.col("ts_hour") >= test_date).cache()

# Splitting train further for tuning
tune_train_df = train_full_df.filter(F.col("ts_hour") < validation_date).cache()
tune_val_df = train_full_df.filter(F.col("ts_hour") >= validation_date).cache()

print(f"Data Ready!")
print(f"Total Training Pool (< Aug): {train_full_df.count():,} rows")
print(f"Testing Data (>= Aug): {test_df.count():,} rows")

# ==========================================
# 2. STRICT TOP 15 FEATURES
# ==========================================
TOP_15_COLS = [
    "radius200m_outflow_lag1", 
    "radius200m_inflow_lag1", 
    "radius100m_inflow_lag1", 
    "station_inflow_lag1", 
    "hod_cos", 
    "radius500m_outflow_lag1", 
    "hod", 
    "radius500m_inflow_lag1", 
    "radius100m_outflow_rollmean6", 
    "radius100m_outflow_lag1", 
    "station_outflow_rollmean6", 
    "radius100m_inflow_rollmean6", 
    "radius100m_outflow_rollmean12", 
    "station_outflow_rollsum6", 
    "radius200m_outflow_rollmean6" 
]

# Because all 15 are numeric, we only need the VectorAssembler!
assembler = VectorAssembler(inputCols=TOP_15_COLS, outputCol="features", handleInvalid="skip")
feature_stages = [assembler]

# ==========================================
# 3. RANDOM FOREST ARCHITECTURE & TUNING
# ==========================================
rf = RandomForestRegressor(featuresCol="features", labelCol=target_col, seed=42)

# The Grid: Test 6 combinations
rf_grid = (ParamGridBuilder()
    .addGrid(rf.maxDepth, [5, 10, 15])      
    .addGrid(rf.numTrees, [50, 100])        
    .build())

evaluator_rmse = RegressionEvaluator(labelCol=target_col, metricName="rmse")
evaluator_mae = RegressionEvaluator(labelCol=target_col, metricName="mae")

print(f"\n--- Tuning OUTFLOW Random Forest ---")
best_rmse = float("inf")
best_params = None

for param_map in rf_grid:
    rf_with_params = rf.copy(param_map)
    pipeline = Pipeline(stages=feature_stages + [rf_with_params])
    
    # Train on past (< July), evaluate on July
    temp_model = pipeline.fit(tune_train_df)
    predictions = temp_model.transform(tune_val_df)
    current_rmse = evaluator_rmse.evaluate(predictions)
    
    if current_rmse < best_rmse:
        best_rmse = current_rmse
        best_params = param_map

print(f"🏆 Best Tuning RMSE (on July Data): {best_rmse:.3f}")

# ==========================================
# 4. FINAL TRAINING & TEST SET EVALUATION
# ==========================================
print(f"\nRetraining final optimized RF on ALL training data (< August)...")
final_rf = rf.copy(best_params)
final_pipeline = Pipeline(stages=feature_stages + [final_rf])

best_final_model = final_pipeline.fit(train_full_df)

# Evaluate on the REAL Future (>= August 1st)
print(f"Testing model on unseen August+ data...")
test_predictions = best_final_model.transform(test_df)

final_rmse = evaluator_rmse.evaluate(test_predictions)
final_mae = evaluator_mae.evaluate(test_predictions)

print("\n==================================")
print("🚀 FINAL TEST SET RESULTS (OUTFLOW)")
print("==================================")
print(f"Test MAE:  {final_mae:.3f} bikes")
print(f"Test RMSE: {final_rmse:.3f} bikes")
print("==================================")

# Clear the cache to free up memory for the save operation
spark.catalog.clearCache()
# Now try saving again
save_path = f"models/{targetStation}/outflow_RF_optimized"
best_final_model.write().overwrite().save(save_path)



Data Ready!
Total Training Pool (< Aug): 20,197 rows
Testing Data (>= Aug): 4,417 rows

--- Tuning OUTFLOW Random Forest ---


26/04/18 01:16:54 WARN DAGScheduler: Broadcasting large task binary with size 1148.8 KiB
26/04/18 01:16:55 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/04/18 01:16:55 WARN DAGScheduler: Broadcasting large task binary with size 3.9 MiB
26/04/18 01:16:55 WARN DAGScheduler: Broadcasting large task binary with size 1059.6 KiB
26/04/18 01:16:56 WARN DAGScheduler: Broadcasting large task binary with size 6.9 MiB
26/04/18 01:16:56 WARN DAGScheduler: Broadcasting large task binary with size 1724.5 KiB
26/04/18 01:16:58 WARN DAGScheduler: Broadcasting large task binary with size 1140.6 KiB
26/04/18 01:16:59 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/04/18 01:16:59 WARN DAGScheduler: Broadcasting large task binary with size 4.0 MiB
26/04/18 01:17:00 WARN DAGScheduler: Broadcasting large task binary with size 1168.9 KiB
26/04/18 01:17:00 WARN DAGScheduler: Broadcasting large task binary with size 7.5 MiB
26/04/18 01:17:01 WARN DAGScheduler: Br

🏆 Best Tuning RMSE (on July Data): 7.338

Retraining final optimized RF on ALL training data (< August)...


26/04/18 01:17:52 WARN DAGScheduler: Broadcasting large task binary with size 1114.4 KiB
26/04/18 01:17:53 WARN DAGScheduler: Broadcasting large task binary with size 2.0 MiB
26/04/18 01:17:54 WARN DAGScheduler: Broadcasting large task binary with size 4.0 MiB
26/04/18 01:17:54 WARN DAGScheduler: Broadcasting large task binary with size 1174.2 KiB
26/04/18 01:17:55 WARN DAGScheduler: Broadcasting large task binary with size 7.5 MiB
26/04/18 01:17:56 WARN DAGScheduler: Broadcasting large task binary with size 2.0 MiB
26/04/18 01:17:56 WARN DAGScheduler: Broadcasting large task binary with size 13.6 MiB
26/04/18 01:17:58 WARN DAGScheduler: Broadcasting large task binary with size 3.4 MiB
26/04/18 01:17:59 WARN DAGScheduler: Broadcasting large task binary with size 23.3 MiB
26/04/18 01:18:01 WARN DAGScheduler: Broadcasting large task binary with size 5.0 MiB
26/04/18 01:18:02 WARN DAGScheduler: Broadcasting large task binary with size 37.2 MiB
26/04/18 01:18:04 WARN DAGScheduler: Broadcas

Testing model on unseen August+ data...

🚀 FINAL TEST SET RESULTS (OUTFLOW)
Test MAE:  3.380 bikes
Test RMSE: 5.352 bikes


26/04/18 01:18:27 WARN TaskSetManager: Stage 14084 contains a task of very large size (16780 KiB). The maximum recommended task size is 1000 KiB.


# Optimising best inflow model (RF)

In [13]:
#########
# FEATURE IMPORTANCE ANALYSIS (For the best RF model)
#########

import pandas as pd
from pyspark.ml import PipelineModel

import pandas as pd
from pyspark.ml import PipelineModel

# 1. Load the model
model_path = f"models/{targetStation}/inflow_RF_optimized"
print(f"Loading model from: {model_path}...")
loaded_pipeline = PipelineModel.load(model_path)
ml_model = loaded_pipeline.stages[-1]

# 2. Extract the importances array
importances = ml_model.featureImportances.toArray()

# 3. THE FIX: Extract true feature names from PySpark metadata
# We push 1 row of data through the pipeline to generate the metadata
# Note: Using 'tune_val_df' here, but you can use 'train_df' or any df in memory
sample_transformed = loaded_pipeline.transform(tune_val_df.limit(1))

# Get the exact column the model used (usually "features")
features_col_name = ml_model.getFeaturesCol()

# Dig into PySpark's hidden metadata dictionary
feature_attrs = sample_transformed.schema[features_col_name].metadata["ml_attr"]["attrs"]

# Unpack the metadata into a list of (index, name) tuples
extracted_names = []
for attr_group in feature_attrs.values():
    for attr in attr_group:
        extracted_names.append((attr['idx'], attr['name']))

# Sort strictly by index so it matches the importances array perfectly!
extracted_names.sort(key=lambda x: x[0])
true_feature_names = [name for idx, name in extracted_names]

# 4. Build a clean Pandas DataFrame (Now they will be the exact same length!)
importance_df = pd.DataFrame({
    "Feature": true_feature_names,
    "Importance": importances
})

# 5. Calculate the percentage and display
importance_df["Importance_%"] = (importance_df["Importance"] * 100).round(2)
importance_df = importance_df.sort_values(by="Importance", ascending=False).reset_index(drop=True)

print("\n🏆 Top 10 Most Important Features:")
display(importance_df.head(10))

print("\n🗑️ The 'Noise' (Bottom 15 Features):")
display(importance_df.tail(15))

Loading model from: models/STN_0001/inflow_RF_optimized...

🏆 Top 10 Most Important Features:


,Feature,Importance,Importance_%
0,radius200m_outflow_lag1,0.180736,18.07
1,radius500m_outflow_lag1,0.168004,16.80
2,radius100m_inflow_lag1,0.162508,16.25
3,station_inflow_lag1,0.095207,9.52
4,hod,0.039330,3.93
5,radius200m_inflow_lag1,0.037813,3.78
6,radius100m_outflow_lag1,0.036362,3.64
7,hod_cos,0.035617,3.56
8,radius500m_inflow_lag1,0.034496,3.45
9,radius100m_inflow_rollmean6,0.023552,2.36



🗑️ The 'Noise' (Bottom 15 Features):


,Feature,Importance,Importance_%
52,radius200m_inflow_rollsum6,8.977484e-04,0.09
53,moy,8.374717e-04,0.08
54,radius200m_outflow_rollsum6,5.919322e-04,0.06
55,precip_rollmean3,3.722940e-04,0.04
56,precip_rollsum3,3.643537e-04,0.04
57,precip,3.624316e-04,0.04
58,temp_bin_vec_15:20,2.768986e-04,0.03
59,temp_bin_vec_10:15,2.232034e-04,0.02
60,temp_bin_vec_0:10,1.869151e-04,0.02
61,temp_bin_vec_20:25,1.708936e-04,0.02


In [14]:
# Extract the top 15 feature names from your ranked dataframe
top_15_features = importance_df["Feature"].head(15).tolist()

print("🎯 Your Top 15 Winning Features:")
print("TOP_15_COLS =", top_15_features)

🎯 Your Top 15 Winning Features:
TOP_15_COLS = ['radius200m_outflow_lag1', 'radius500m_outflow_lag1', 'radius100m_inflow_lag1', 'station_inflow_lag1', 'hod', 'radius200m_inflow_lag1', 'radius100m_outflow_lag1', 'hod_cos', 'radius500m_inflow_lag1', 'radius100m_inflow_rollmean6', 'station_outflow_lag1', 'station_inflow_rollmean6', 'radius100m_inflow_rollsum6', 'station_inflow_rollsum6', 'radius500m_inflow_lag12']


In [15]:
# ==========================================
# 1. SETUP & IMPORTS
# ==========================================
from pyspark.ml.regression import RandomForestRegressor # Swapped to RF
from pyspark.ml.tuning import ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

targetStation = "STN_0001" 
target_col = "station_inflow" 

# Filter and Split logic (Remains identical to your previous run)
gold_df_stn = gold_df.filter(F.col("station_id") == targetStation)
test_date = "2025-08-01 00:00:00"
validation_date = "2025-07-01 00:00:00"

train_full_df = gold_df_stn.filter(F.col("ts_hour") < test_date).cache()
test_df = gold_df_stn.filter(F.col("ts_hour") >= test_date).cache()

tune_train_df = train_full_df.filter(F.col("ts_hour") < validation_date).cache()
tune_val_df = train_full_df.filter(F.col("ts_hour") >= validation_date).cache()

# ==========================================
# 2. FEATURE SELECTION (INFLOW)
# ==========================================
TOP_15_INFLOW_COLS = [
    "radius200m_inflow_lag1", 
    "radius500m_outflow_lag1", 
    "radius100m_inflow_lag1", 
    "station_inflow_lag1", 
    "hod", 
    "radius200m_inflow_lag1", 
    "radius100m_outflow_lag1", 
    "hod_cos", 
    "radius500m_inflow_lag1", 
    "radius100m_inflow_rollmean6", 
    "station_outflow_lag1", 
    "station_inflow_rollmean6", 
    "radius100m_inflow_rollsum6", 
    "station_inflow_rollsum6", 
    "radius500m_inflow_lag12"
]

assembler = VectorAssembler(inputCols=TOP_15_INFLOW_COLS, outputCol="features", handleInvalid="skip")
feature_stages = [assembler]

# ==========================================
# 3. RF ARCHITECTURE & TUNING
# ==========================================
# Initialize Random Forest
rf = RandomForestRegressor(featuresCol="features", labelCol=target_col, seed=42)

# RF Grids focus on maxDepth and numTrees
rf_grid = (ParamGridBuilder()
    .addGrid(rf.maxDepth, [5, 10])     # Tree depth
    .addGrid(rf.numTrees, [50, 100])   # Number of trees in the forest
    .build())

evaluator_rmse = RegressionEvaluator(labelCol=target_col, metricName="rmse")
evaluator_mae = RegressionEvaluator(labelCol=target_col, metricName="mae")

print(f"\n--- Tuning INFLOW Random Forest ---")
best_rmse = float("inf")
best_params = None

for param_map in rf_grid:
    rf_with_params = rf.copy(param_map)
    pipeline = Pipeline(stages=feature_stages + [rf_with_params])
    
    temp_model = pipeline.fit(tune_train_df)
    predictions = temp_model.transform(tune_val_df)
    current_rmse = evaluator_rmse.evaluate(predictions)
    
    if current_rmse < best_rmse:
        best_rmse = current_rmse
        best_params = param_map

# ==========================================
# 4. FINAL TRAINING & SAVING
# ==========================================
print(f"🏆 Best RF Params Found. RMSE: {best_rmse:.3f}")

# Final train on all data < Aug 1st
final_rf = rf.copy(best_params)
final_pipeline = Pipeline(stages=feature_stages + [final_rf])
best_final_inflow_model = final_pipeline.fit(train_full_df)

# Evaluate on August+ Data
test_predictions = best_final_inflow_model.transform(test_df)
print(f"Final Test MAE: {evaluator_mae.evaluate(test_predictions):.3f}")

# Memory safeguard: clear cache before saving
spark.catalog.clearCache()

# Save as inflow_RF_optimized
save_path = f"models/{targetStation}/inflow_RF_optimized"
best_final_inflow_model.write().overwrite().save(save_path)
print(f"✅ Model saved to: {save_path}")


--- Tuning INFLOW Random Forest ---


26/04/18 01:23:36 WARN DAGScheduler: Broadcasting large task binary with size 1150.0 KiB
26/04/18 01:23:37 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/04/18 01:23:37 WARN DAGScheduler: Broadcasting large task binary with size 3.9 MiB
26/04/18 01:23:38 WARN DAGScheduler: Broadcasting large task binary with size 1060.2 KiB
26/04/18 01:23:38 WARN DAGScheduler: Broadcasting large task binary with size 6.9 MiB
26/04/18 01:23:38 WARN DAGScheduler: Broadcasting large task binary with size 1742.7 KiB
26/04/18 01:23:40 WARN DAGScheduler: Broadcasting large task binary with size 1143.2 KiB
26/04/18 01:23:41 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/04/18 01:23:42 WARN DAGScheduler: Broadcasting large task binary with size 4.0 MiB
26/04/18 01:23:42 WARN DAGScheduler: Broadcasting large task binary with size 1175.0 KiB
26/04/18 01:23:43 WARN DAGScheduler: Broadcasting large task binary with size 7.5 MiB
26/04/18 01:23:44 WARN DAGScheduler: Br

🏆 Best RF Params Found. RMSE: 6.792


26/04/18 01:23:48 WARN DAGScheduler: Broadcasting large task binary with size 1116.5 KiB
26/04/18 01:23:49 WARN DAGScheduler: Broadcasting large task binary with size 2.0 MiB
26/04/18 01:23:49 WARN DAGScheduler: Broadcasting large task binary with size 4.0 MiB
26/04/18 01:23:50 WARN DAGScheduler: Broadcasting large task binary with size 1181.0 KiB
26/04/18 01:23:50 WARN DAGScheduler: Broadcasting large task binary with size 7.5 MiB
26/04/18 01:23:52 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/04/18 01:23:52 WARN DAGScheduler: Broadcasting large task binary with size 13.8 MiB
26/04/18 01:23:54 WARN DAGScheduler: Broadcasting large task binary with size 3.4 MiB


Final Test MAE: 2.906


26/04/18 01:23:55 WARN TaskSetManager: Stage 14215 contains a task of very large size (2821 KiB). The maximum recommended task size is 1000 KiB.


✅ Model saved to: models/STN_0001/inflow_RF_optimized
